# 03 — Quantitative Evaluation

This notebook is the interactive companion to `scripts/evaluate.py`. It defines and runs each metric one at a time so you can inspect intermediate values rather than just the final JSON.

Metrics computed:
1. **Self-recommendation precision** — for random postings, is the true role in top-k?
2. **Skill-extractor agreement** — does our regex extractor recover the skills humans listed in `skills_desc`?
3. **Next-skill prediction** — given partial skills, do we suggest the missing one as a top-N learning target?
4. **Coverage and diversity** over 500 random profiles
5. **Latency**

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.recommender import CareerRecommender
from scripts.evaluate import (
    self_recommendation_precision,
    skill_extractor_agreement,
    next_skill_prediction,
    coverage_and_diversity,
    latency,
)

rec = CareerRecommender.load_or_build()
print(f'Loaded recommender: {len(rec.roles)} roles, {rec.tfidf.vocabulary_size()} skill vocab, has_postings={rec.postings is not None}')

## 1. Self-recommendation precision
Random baseline at top-k: P@k ≈ k / 1060. We expect to be **dramatically** above that.

In [ ]:
sr = self_recommendation_precision(rec, n_samples=2000)
print(json.dumps(sr, indent=2))
print('Multipliers vs random baseline:')
for k in (1, 5, 10):
    print(f'  P@{k}: {sr[f"precision_at_{k}"]:.3f}  ({sr[f"precision_at_{k}"] / (k/1060):.0f}x random)')

## 2. Skill-extractor agreement vs. human-curated `skills_desc`
Soft ground truth — `skills_desc` is itself free text, so this benchmarks our regex against a recruiter's prose, not a perfect taxonomy. We expect mean recall > mean precision (we should miss more often than we hallucinate).

In [ ]:
sea = skill_extractor_agreement(rec)
print(json.dumps(sea, indent=2))

## 3. Next-skill prediction
For each posting with ≥4 skills, hide one random skill, query with the rest, and check whether the held-out skill appears in the union of top-N missing skills across the top-5 recommended roles. This directly tests the explainability story: 'what should I learn next?'

In [ ]:
nsp = next_skill_prediction(rec, n_samples=1000)
print(json.dumps(nsp, indent=2))
print('Multipliers vs random baseline (5/121 ≈ 4.1%):')
for k in (3, 5, 10):
    print(f'  hits@{k}: {nsp[f"next_skill_at_{k}"]:.3f}  ({nsp[f"next_skill_at_{k}"] / (k/121):.1f}x random)')

## 4. Coverage and diversity
Coverage = fraction of the role index ever surfaced across 500 random user profiles. Diversity = mean intra-list pairwise Jaccard distance among top-5 role skill-sets.

In [ ]:
cov = coverage_and_diversity(rec, n_profiles=500)
print(json.dumps(cov, indent=2))

## 5. Latency

In [ ]:
lat = latency(rec, n_calls=200)
print(json.dumps(lat, indent=2))

## Final summary

All numbers in `docs/REPORT.md` are produced by `scripts/evaluate.py` (random seed = 42).